# Semana 7 · Sesión 1: Marcos de referencia, vectores y diádicas

**Módulo 2**

## Objetivos de la sesión

1. Construir marcos de referencia con `ReferenceFrame` y operar vectores
   —suma, producto punto, producto cruz— sin escoger coordenadas antes de
   tiempo.
2. Orientar un marco respecto a otro y leer su matriz de cosenos directores
   como una rotación: la matriz ortogonal de la semana 6.
3. Derivar vectores en el tiempo desde marcos distintos, usar la velocidad
   angular, y representar la inercia de un cuerpo como una diádica.

## Retomamos

El Módulo 1 terminó con un sistema físico resuelto de punta a punta: matrices,
eigenvalores, modos normales, gráfica y reporte. Todo con las herramientas
generales de SymPy — `Symbol`, `diff`, `solve`, `Matrix`.

Hoy empieza el Módulo 2, y cambiamos de nivel: SymPy trae **submódulos de
física** que ya saben lo que es un vector, un marco de referencia o un cuerpo
rígido. El primero es `sympy.physics.mechanics`, y el concepto con el que
arranca todo es el de **marco de referencia**. Las matrices de la semana pasada
reaparecen, pero con un papel nuevo: como **rotaciones** entre marcos.

`sympy.physics.mechanics` se importa aparte y, por convención de su propia
documentación, con el alias `me`. Trae además su propia versión de
`init_printing`: `me.init_vprinting()` hace lo mismo que `sp.init_printing()`
y agrega la **notación de punto** de la mecánica — $\dot q$ en lugar de
$\frac{d}{dt}q(t)$. Por eso esta semana la llamamos en su lugar.

`Math` sirve para mostrar LaTeX escrito a mano; en un momento vemos para qué.

In [ ]:
import sympy as sp
import sympy.physics.mechanics as me
from IPython.display import Math

me.init_vprinting()   # init_printing + notación de punto para derivadas en t

## `ReferenceFrame`: un marco trae su base

Un **marco de referencia** es un punto de vista desde el que se describe el
movimiento: el laboratorio, un carrusel que gira, un péndulo que oscila. En
`mechanics` se crea con `me.ReferenceFrame("N")`, y cada marco trae sus tres
vectores unitarios, ortogonales entre sí:

| Vector unitario | Escribe |
|---|---|
| $\hat{n}_x$ | `N.x` |
| $\hat{n}_y$ | `N.y` |
| $\hat{n}_z$ | `N.z` |

La letra `N` es costumbre: el marco **n**ewtoniano, el del laboratorio, donde
valen las leyes de Newton.

In [ ]:
N = me.ReferenceFrame("N")

display(N.x)
display(N.y)
display(N.z)

## Vectores: combinaciones de la base

Un vector se escribe como combinación de esos unitarios, con coeficientes que
son expresiones de SymPy cualesquiera. Y aquí está la idea central de la
sesión: un `Vector` de `mechanics` **no es una columna de números**. Es un
objeto geométrico que *sabe en qué marco* está escrito cada pedazo. Las
componentes aparecen hasta que uno las pide, y se piden **respecto a un
marco**.

In [ ]:
a1, a2, a3 = sp.symbols("a1 a2 a3", real=True)
b1, b2, b3 = sp.symbols("b1 b2 b3", real=True)

a = a1*N.x + a2*N.y + a3*N.z
b = b1*N.x + b2*N.y + b3*N.z

display(a + b)
display(3*a)

## Operaciones

| Operación | Escribe | Devuelve |
|---|---|---|
| Producto punto | `a.dot(b)` | Un escalar (expresión de SymPy) |
| Producto cruz | `a.cross(b)` | Un `Vector` |
| Magnitud | `a.magnitude()` | Un escalar |
| Vector unitario | `a.normalize()` | Un `Vector` |
| Componentes en un marco | `a.to_matrix(N)` | Una `sp.Matrix` columna |

Fíjate en la columna de la derecha: el producto punto y la magnitud **salen del
mundo de los vectores** y devuelven una expresión de SymPy común y corriente,
que ya sabemos etiquetar con `sp.Eq`.

Un vector, en cambio, **no** se deja etiquetar con `sp.Eq`: no es una expresión
de SymPy, y `sp.Eq` no lo acepta (abajo lo vemos fallar, a propósito). La
salida es escribir la etiqueta en LaTeX y pedirle a `mechanics` el LaTeX del
vector con `me.vlatex`; `Math` junta las dos cosas.

In [ ]:
display(sp.Eq(sp.Symbol(r"\mathbf{a} \cdot \mathbf{b}"), a.dot(b)))
display(sp.Eq(sp.Symbol(r"|\mathbf{a}|"), a.magnitude()))

try:
    sp.Eq(sp.Symbol("a"), a)
except Exception as error:
    print(type(error).__name__, "-", error)

display(Math(r"\mathbf{a} \times \mathbf{b} = " + me.vlatex(a.cross(b))))

## Física: torque y trabajo

Una llave de tuercas de longitud $\ell$, acostada sobre el eje $\hat{n}_x$. En
su extremo aplicamos una fuerza de magnitud $F$ que forma un ángulo $\varphi$
con la llave. El **torque** respecto a la tuerca es
$\boldsymbol{\tau} = \mathbf{r} \times \mathbf{F}$.

Y si esa misma fuerza empuja un objeto una distancia $d$ a lo largo de
$\hat{n}_x$, el **trabajo** es $W = \mathbf{F} \cdot \mathbf{d}$.

Antes de correr la celda, predice: ¿hacia dónde apunta el torque, y para qué
$\varphi$ es máximo? Fíjate además en lo que **no** vamos a escribir: ninguna
columna de componentes ni la regla del determinante para el producto cruz.

In [ ]:
ell, fuerza_F, d = sp.symbols("ell F d", positive=True)   # longitudes y magnitud
phi = sp.Symbol("varphi", real=True)                      # un ángulo cualquiera

brazo = ell * N.x
fuerza = fuerza_F*sp.cos(phi)*N.x + fuerza_F*sp.sin(phi)*N.y

display(Math(r"\boldsymbol{\tau} = " + me.vlatex(brazo.cross(fuerza))))
display(sp.Eq(sp.Symbol("W"), fuerza.dot(d*N.x)))

## TODO en clase 1

Una partícula de carga $q$ se mueve en el plano con velocidad
$\mathbf{v} = v_x\,\hat{n}_x + v_y\,\hat{n}_y$ dentro de un campo magnético
uniforme $\mathbf{B} = B_0\,\hat{n}_z$. La **fuerza de Lorentz** es
$\mathbf{F} = q\,\mathbf{v} \times \mathbf{B}$.

1. Declara `carga` (nombre de símbolo `"q"`), `v_x`, `v_y` y `B_0`. Piensa las
   suposiciones: ¿una carga es positiva? ¿y una componente de velocidad?
2. Arma `velocidad`, `campo` y `fuerza_lorentz` como vectores en `N`, y muestra
   la fuerza con su etiqueta.
3. Comprueba que la fuerza es **perpendicular** a la velocidad: el producto
   punto, simplificado, tiene que dar cero. ¿Qué dice eso sobre el trabajo que
   hace un campo magnético?
4. Calcula la magnitud de la fuerza. Debe salirte
   $B_0\,|q|\sqrt{v_x^2 + v_y^2}$. ¿De dónde sale el valor absoluto?

In [ ]:
# TODO en clase: la fuerza de Lorentz
carga = ...
v_x = ...
v_y = ...
B_0 = ...

velocidad = ...
campo = ...
fuerza_lorentz = ...

## `dynamicsymbols`: cantidades que cambian en el tiempo

En mecánica casi todo depende del tiempo: ángulos, posiciones, velocidades.
`me.dynamicsymbols("q")` crea una función $q(t)$ lista para derivar, que con
`init_vprinting` se muestra con su punto.

Un detalle que conecta con la semana 4: el tiempo de `mechanics` es
`sp.Symbol("t")`, **sin suposiciones**. Si declaramos nuestro propio `t` con
`real=True`, para SymPy es otro símbolo distinto —igual que $x$ real y $x$ sin
suposiciones no son el mismo—, y derivar respecto a él daría cero.

In [ ]:
t = sp.Symbol("t")          # el mismo tiempo que usa mechanics: sin suposiciones
q = me.dynamicsymbols("q")

display(q)
display(q.diff(t))          # la derivada, ya con su punto
display(q.diff(t, 2))

# Un t real sería otro símbolo: q no depende de él.
t_real = sp.Symbol("t", real=True)
display(sp.Eq(sp.Derivative(q, t_real), q.diff(t_real)))

## Orientar un marco

Un marco se vuelve interesante cuando se mueve respecto a otro. El caso más
simple: un marco $B$ que es $N$ **girado un ángulo $q$ alrededor de
$\hat{n}_z$**.

![El marco B es N girado un ángulo q alrededor del eje z](img/marco-rotado.svg)

Se declara con `B.orient_axis(N, N.z, q)`: "orienta a $B$ respecto a $N$,
girando alrededor de `N.z`, un ángulo `q`". A partir de ahí `mechanics` sabe
convertir entre las dos bases, en ambas direcciones.

In [ ]:
B = me.ReferenceFrame("B")
B.orient_axis(N, N.z, q)

display(Math(r"\hat{b}_x = " + me.vlatex(B.x.express(N))))
display(Math(r"\hat{b}_y = " + me.vlatex(B.y.express(N))))

## La matriz de cosenos directores

Toda la información de la orientación cabe en una matriz de 3×3, la **matriz
de cosenos directores** (DCM, por sus siglas en inglés): `B.dcm(N)`. Cada
renglón es un unitario de $B$ escrito en la base de $N$, y cada entrada es el
coseno del ángulo entre dos unitarios — de ahí el nombre.

Es una `sp.Matrix` como las de la semana 6, así que todo lo de entonces aplica.
Una rotación no estira ni refleja nada, y eso se ve en dos propiedades que se
pueden comprobar: la matriz es **ortogonal** ($R R^T = \mathbb{1}$, su inversa
es su transpuesta) y su determinante es $1$.

In [ ]:
rotacion = B.dcm(N)
display(sp.Eq(sp.Symbol("{}^B R^N"), rotacion, evaluate=False))

display(sp.Eq(sp.Symbol("R R^T"), sp.simplify(rotacion * rotacion.T), evaluate=False))
display(sp.Eq(sp.Symbol(r"\det R"), sp.simplify(rotacion.det())))

## Un vector, dos bases

`express` reescribe un vector en la base de otro marco: el vector es el mismo,
cambian las componentes. Y las operaciones entre vectores de marcos distintos
funcionan sin convertir nada a mano: `mechanics` busca la cadena de
orientaciones y hace la cuenta.

Además de `orient_axis` hay otras formas de orientar, que conviene saber que
existen:

| Método | Para |
|---|---|
| `orient_axis(N, eje, angulo)` | Una rotación alrededor de un eje (hoy) |
| `orient_body_fixed(N, (a, b, c), "123")` | Tres rotaciones sucesivas: ángulos de Euler |
| `orient_explicit(N, matriz)` | Dar la DCM directamente |

In [ ]:
display(Math(r"\hat{n}_x = " + me.vlatex(N.x.express(B))))

# Producto punto entre unitarios de marcos distintos: el coseno del ángulo.
display(sp.Eq(sp.Symbol(r"\hat{n}_x \cdot \hat{b}_x"), N.x.dot(B.x)))
display(sp.Eq(sp.Symbol(r"\hat{n}_x \cdot \hat{b}_y"), N.x.dot(B.y)))

## TODO en clase 2

Dos rotaciones sucesivas, con ángulos fijos $\alpha$ y $\beta$ (símbolos
reales, no `dynamicsymbols`):

- `C`: $N$ girado $\alpha$ alrededor de `N.z`.
- `D`: $C$ girado $\beta$ alrededor de `C.x`.

1. Crea los dos marcos y obtén `rotacion_total = D.dcm(N)`. Comprueba que es
   igual al producto `D.dcm(C) * C.dcm(N)`: la diferencia, simplificada, tiene
   que ser la matriz cero. **Las rotaciones se componen multiplicando sus
   matrices.**
2. Comprueba que `rotacion_total` es ortogonal.
3. Ahora en el orden inverso: `E` es $N$ girado $\beta$ alrededor de `N.x`, y
   `F` es $E$ girado $\alpha$ alrededor de `E.z`. Calcula
   `sp.simplify(D.dcm(N) - F.dcm(N))`. ¿Sale cero? ¿Qué propiedad de la
   semana 6 estás viendo, y qué significa físicamente al girar un libro?

In [ ]:
# TODO en clase: rotaciones sucesivas
alfa = ...
beta = ...

C = ...
D = ...

rotacion_total = ...

## Velocidad angular

Como orientamos $B$ con un ángulo que **depende del tiempo**, $B$ gira
respecto a $N$. `mechanics` calcula la velocidad angular por su cuenta, a
partir de la orientación: `B.ang_vel_in(N)`. Para una rotación alrededor de un
eje fijo, es lo esperado: $\dot q$ a lo largo de ese eje.

In [ ]:
omega_b = B.ang_vel_in(N)
display(Math(r"{}^N\boldsymbol{\omega}^B = " + me.vlatex(omega_b)))

## Derivar un vector: ¿desde qué marco?

Pregunta con trampa: ¿cuánto vale la derivada temporal de $\hat{b}_x$? Depende
de quién mire. Para alguien parado en $B$, $\hat{b}_x$ no se mueve: su
derivada es cero. Para alguien en $N$, $\hat{b}_x$ gira con el marco, y su
derivada no es cero.

Por eso en `mechanics` una derivada temporal **siempre** lleva marco:
`v.dt(N)`. Las dos derivadas se relacionan con el **teorema de transporte**:

$$\frac{{}^N d\mathbf{u}}{dt} = \frac{{}^B d\mathbf{u}}{dt}
  + {}^N\boldsymbol{\omega}^B \times \mathbf{u}$$

que es de donde van a salir, la próxima sesión, las aceleraciones centrípeta y
de Coriolis.

In [ ]:
display(B.x.dt(B))   # desde B: no cambia
display(B.x.dt(N))   # desde N: gira

# El teorema de transporte, comprobado para un vector cualquiera escrito en B.
u1, u2 = me.dynamicsymbols("u1 u2")
u = u1*B.x + u2*B.y

diferencia = u.dt(N) - (u.dt(B) + omega_b.cross(u))
display(diferencia.express(N).simplify())

## Diádicas

Hay cantidades físicas que no son ni escalares ni vectores, sino **operadores
que convierten un vector en otro**: el tensor de inercia convierte la velocidad
angular en momento angular, el de esfuerzos convierte una normal en una fuerza.
En `mechanics` se llaman **diádicas**.

La pieza básica es el producto externo $\hat{n}_x \otimes \hat{n}_y$
(`me.outer(N.x, N.y)`), que actúa sobre un vector con el producto punto:

$$(\hat{n}_x \otimes \hat{n}_y) \cdot \mathbf{v} = \hat{n}_x\,(\hat{n}_y \cdot \mathbf{v})$$

"Toma la componente $y$ de $\mathbf{v}$ y ponla a lo largo de $\hat{n}_x$".

In [ ]:
diadica = me.outer(N.x, N.y)
display(diadica)

display(diadica.dot(N.y))   # componente y = 1  ->  n_x
display(diadica.dot(N.x))   # componente y = 0  ->  vector cero

## Física: momento angular $\mathbf{H} = \mathbf{I} \cdot \boldsymbol{\omega}$

Un cuerpo rígido con momentos de inercia principales $I_1, I_2, I_3$ a lo
largo de los ejes de $B$. Su tensor de inercia es la diádica
`me.inertia(B, I1, I2, I3)`, y su momento angular es
$\mathbf{H} = \mathbf{I} \cdot \boldsymbol{\omega}$.

Si el cuerpo gira alrededor de un eje principal, $\mathbf{H}$ es paralelo a
$\boldsymbol{\omega}$. Pero si gira alrededor de un eje **inclinado**, con
$I_1 \neq I_3$, deja de serlo — y eso, en una llanta mal balanceada, es lo que
hace temblar el volante.

In [ ]:
I1, I2, I3 = sp.symbols("I1 I2 I3", positive=True)   # momentos de inercia
w1, w3 = sp.symbols("omega1 omega3", real=True)

inercia = me.inertia(B, I1, I2, I3)
display(inercia)

giro_inclinado = w1*B.x + w3*B.z
momento_angular = inercia.dot(giro_inclinado)
display(Math(r"\mathbf{H} = " + me.vlatex(momento_angular)))

# Si H fuera paralelo a omega, este producto cruz sería cero.
display(Math(r"\mathbf{H} \times \boldsymbol{\omega} = "
             + me.vlatex(momento_angular.cross(giro_inclinado))))

## De vuelta a los eigenvectores

¿Y si el cuerpo no viene con sus ejes principales ya alineados con el marco?
Entonces la matriz de inercia tiene entradas fuera de la diagonal, y los ejes
principales hay que **encontrarlos**. `to_matrix(N)` convierte la diádica en su
`sp.Matrix`, y a partir de ahí es exactamente la semana 6: los ejes principales
son los **eigenvectores** y los momentos principales, los **eigenvalores**.

In [ ]:
I0, Ip = sp.symbols("I_0 I_p", positive=True)

inercia_n = me.inertia(N, I0, I0, 2*I0, ixy=-Ip)   # entrada xy de la matriz: -I_p
matriz_inercia = inercia_n.to_matrix(N)
display(matriz_inercia)

for valor, _, vectores in matriz_inercia.eigenvects():
    print("momento principal:")
    display(valor)
    print("eje principal:")
    display(vectores[0].T)

Los ejes principales son $(0, 0, 1)$ y las dos diagonales $(1, 1, 0)$ y
$(-1, 1, 0)$ del plano $xy$ — los mismos vectores que los modos normales de las
dos masas de la semana pasada. No es casualidad: en los dos casos es una matriz
simétrica con la misma estructura, y la física de "girar sin temblar" y la de
"oscilar sin mezclarse" es la misma matemática.

## Resumen

Hoy entramos a `sympy.physics.mechanics`. Un `ReferenceFrame` trae su base, y
un `Vector` es una combinación de esas bases que sabe en qué marco está escrito:
`dot`, `cross` y `magnitude` funcionan sin escoger coordenadas, y `express` o
`to_matrix` dan las componentes cuando hacen falta. Como un vector no es una
expresión de SymPy, se etiqueta con `Math` y `me.vlatex`.

`orient_axis` coloca un marco respecto a otro; su DCM es una matriz ortogonal
de determinante 1, y las rotaciones se componen multiplicando —y no conmutan—.
Con ángulos que dependen del tiempo aparece la velocidad angular, y con ella la
lección más importante del día: **una derivada temporal siempre es respecto a
un marco**, y el teorema de transporte conecta dos marcos. Las diádicas
completan el cuadro: la inercia es un operador, y sus ejes principales son
eigenvectores.

**Siguiente sesión:** agregamos **puntos**. Con posiciones, velocidades y
aceleraciones de puntos haremos cinemática de partículas —de ahí saldrán las
aceleraciones centrípeta y de Coriolis— y de cuerpos rígidos.